# Data Preparation for Machine Learning → Linear Regression Model
This notebook shows an end-to-end workflow to make data **ML-ready** and build a **Linear Regression** model.

**You will learn:**
- Load & inspect data  
- feature engineering  
- Handling missing values (imputation)  
- Encoding categorical columns 
- Scaling numeric columns  
- Train/Test split  
- Model training (Linear Regression)  
- Evaluation using **MSE** and **RMSE**  


## 1. Imports

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

## 2. Load dataset

In [4]:
# Dataset file path
DATA_PATH = "../data/melb_data.csv"

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

Shape: (13580, 21)


,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,4/03/2017,2.5,3067.0,...,2.0,1.0,94.0,NaN,NaN,Yarra,-37.7969,144.9969,Northern Metropolitan,4019.0
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,4/06/2016,2.5,3067.0,...,1.0,2.0,120.0,142.0,2014.0,Yarra,-37.8072,144.9941,Northern Metropolitan,4019.0


## 3. Quick inspection

In [5]:
# info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         13580 non-null  object 
 1   Address        13580 non-null  object 
 2   Rooms          13580 non-null  int64  
 3   Type           13580 non-null  object 
 4   Price          13580 non-null  float64
 5   Method         13580 non-null  object 
 6   SellerG        13580 non-null  object 
 7   Date           13580 non-null  object 
 8   Distance       13580 non-null  float64
 9   Postcode       13580 non-null  float64
 10  Bedroom2       13580 non-null  float64
 11  Bathroom       13580 non-null  float64
 12  Car            13518 non-null  float64
 13  Landsize       13580 non-null  float64
 14  BuildingArea   7130 non-null   float64
 15  YearBuilt      8205 non-null   float64
 16  CouncilArea    12211 non-null  object 
 17  Lattitude      13580 non-null  float64
 18  Longti

In [6]:
# describe
df.describe()

,Rooms,Price,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude,Propertycount
count,13580.000000,1.358000e+04,13580.000000,13580.000000,13580.000000,13580.000000,13518.000000,13580.000000,7130.000000,8205.000000,13580.000000,13580.000000,13580.000000
mean,2.937997,1.075684e+06,10.137776,3105.301915,2.914728,1.534242,1.610075,558.416127,151.967650,1964.684217,-37.809203,144.995216,7454.417378
std,0.955748,6.393107e+05,5.868725,90.676964,0.965921,0.691712,0.962634,3990.669241,541.014538,37.273762,0.079260,0.103916,4378.581772
min,1.000000,8.500000e+04,0.000000,3000.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1196.000000,-38.182550,144.431810,249.000000
25%,2.000000,6.500000e+05,6.100000,3044.000000,2.000000,1.000000,1.000000,177.000000,93.000000,1940.000000,-37.856822,144.929600,4380.000000
50%,3.000000,9.030000e+05,9.200000,3084.000000,3.000000,1.000000,2.000000,440.000000,126.000000,1970.000000,-37.802355,145.000100,6555.000000
75%,3.000000,1.330000e+06,13.000000,3148.000000,3.000000,2.000000,2.000000,651.000000,174.000000,1999.000000,-37.756400,145.058305,10331.000000
max,10.000000,9.000000e+06,48.100000,3977.000000,20.000000,8.000000,10.000000,433014.000000,44515.000000,2018.000000,-37.408530,145.526350,21650.000000


## 4. Define target and features

In [7]:
df.head(2)

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0


In [9]:
# Target
y = df['Price']

# Features
X = df.drop(columns = ['Price'])

In [10]:
X.head(2)

,Suburb,Address,Rooms,Type,Method,SellerG,Date,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,S,Biggin,3/12/2016,2.5,3067.0,2.0,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,S,Biggin,4/02/2016,2.5,3067.0,2.0,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0


In [11]:
X.drop(columns = ['Address', 'Propertycount'], axis = 1, inplace = True)

In [12]:
X.drop(columns = ['Lattitude', 'Longtitude','SellerG'], axis = 1, inplace = True)

In [13]:
X.columns

Index(['Suburb', 'Rooms', 'Type', 'Method', 'Date', 'Distance', 'Postcode',
       'Bedroom2', 'Bathroom', 'Car', 'Landsize', 'BuildingArea', 'YearBuilt',
       'CouncilArea', 'Regionname'],
      dtype='object')

## 5. Feature engineering (Date → Year/Month/DayOfWeek)

In [14]:
# Convert Date to datetime and extract useful components
X['Date'] = pd.to_datetime(X['Date'], errors = "coerce", dayfirst = True)
X['SoldYear'] = X['Date'].dt.year
X['SoldMonth'] = X['Date'].dt.month
X['SoldDay'] = X['Date'].dt.day

# Drop raw Date after extracting features
X.drop(columns = ['Date'], axis = 1, inplace = True)

# Address is high-cardinality text. For a clean baseline, we drop it.
# done

In [15]:
X.head(2)

,Suburb,Rooms,Type,Method,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Regionname,SoldYear,SoldMonth,SoldDay
0,Abbotsford,2,h,S,2.5,3067.0,2.0,1.0,1.0,202.0,NaN,NaN,Yarra,Northern Metropolitan,2016,12,3
1,Abbotsford,2,h,S,2.5,3067.0,2.0,1.0,0.0,156.0,79.0,1900.0,Yarra,Northern Metropolitan,2016,2,4


## 6. Select numeric vs categorical columns

In [18]:
num_X = X.select_dtypes(include = "number")
cat_X = X.select_dtypes(exclude = "number")

In [19]:
num_X.columns

Index(['Rooms', 'Distance', 'Postcode', 'Bedroom2', 'Bathroom', 'Car',
       'Landsize', 'BuildingArea', 'YearBuilt', 'SoldYear', 'SoldMonth',
       'SoldDay'],
      dtype='object')

In [20]:
cat_X.columns

Index(['Suburb', 'Type', 'Method', 'CouncilArea', 'Regionname'], dtype='object')

## 7. Preprocessing pipeline (Impute + Encode + Scale)

In [22]:
#Scaling : converting all numerical features to the same scale
# Imputation : filling in missing values with a strategy (mean, median, most_frequent, constant)
#simple imputer : a transformer that fills in missing values using a specified strategy
# Encoding : converting categorical variables into a format that can be provided to ML algorithms (One-Hot Encoding)
#one-hot encoding : creating binary columns for each category in a categorical feature

In [21]:
# Numeric: median impute + scale
num_transformations = Pipeline(steps = [("imputer", SimpleImputer(strategy = "median")),
                  ("scaling", StandardScaler())])
# Categorical: most_frequent impute + one-hot encode
cat_transformations = Pipeline(steps = [("imputer", SimpleImputer(strategy = "most_frequent")),
                  ("onehot", OneHotEncoder(handle_unknown = 'ignore'))])

preprocessor = ColumnTransformer(transformers = [("num", num_transformations, num_X.columns.tolist()),
                                  ("cat", cat_transformations, cat_X.columns.tolist())],
                  remainder = "drop")
preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


## 8. Train-test split

In [23]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.15)

In [24]:
X_test.shape

(2037, 17)

In [25]:
y_test

6977      320000.0
2520     1375000.0
2739     1450000.0
2487     1425000.0
12993     931000.0
           ...    
2080     1260000.0
4584      401000.0
7144     1300000.0
6678      955000.0
8694     1575000.0
Name: Price, Length: 2037, dtype: float64

## 9. Train Linear Regression model

In [26]:
model = LinearRegression()
final_pipeline = Pipeline(steps = [("preprocessor",preprocessor),
                  ("model", model)])
final_pipeline

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [27]:
final_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## 10. Evaluate with MSE and RMSE

In [28]:
predictions = final_pipeline.predict(X_test)
mean_squared_error(y_test, predictions)
print("Mean Squared Error:", mean_squared_error(y_test, predictions))
root_mse = np.sqrt(mean_squared_error(y_test, predictions))
print("Root Mean Squared Error:", root_mse)

Mean Squared Error: 126280546485.77144
Root Mean Squared Error: 355359.74235381733


## 11. Sample predictions

In [29]:
print(predictions)

[ 762617.13609558 1321603.12868081 1197854.09087295 ... 1686787.23289134
  886108.3139105  1099695.2519455 ]


In [31]:
predictions

array([ 762617.13609558, 1321603.12868081, 1197854.09087295, ...,
       1686787.23289134,  886108.3139105 , 1099695.2519455 ],
      shape=(2037,))

## 12. Test with custom sample input

In [30]:
# Create a sample input with one row
sample = pd.DataFrame([{
    # Numeric columns
    'Rooms': 3,
    'Distance': 10.0,
    'Postcode': 3073.0,
    'Bedroom2': 3.0,
    'Bathroom': 1.0,
    'Car': 2.0,
    'Landsize': 440.0,
    'BuildingArea': 120.0,
    'YearBuilt': 1990.0,
    'SoldYear': 2017,
    'SoldMonth': 5,
    'SoldDay': 13,
    # Categorical columns
    'Suburb': 'Abbotsford',
    'Type': 'h',          # h=house, t=townhouse, u=unit
    'Method': 'S',        # S=sold, SP=sold prior, PI=passed in, VB=vendor bid
    'CouncilArea': 'Yarra',
    'Regionname': 'Northern Metropolitan'
}])

predicted_price = final_pipeline.predict(sample)
print(f"Predicted Price: ${predicted_price[0]:,.2f}")

Predicted Price: $1,284,672.81
